In [ ]:
%run ../src/data/globalvariables

In [ ]:
%run ../src/data/lakehousefunction

In [ ]:
def expect(rule, df, condition, notebook):
    """Return an error_record when rows violate condition, else None."""
    violations = df.filter(f"NOT ({condition})").count()
    print(f"expect {rule}: {violations} violations")
    if violations:
        return error_record(notebook, Exception(f"{rule}: {violations} rows violate [{condition}]"))

In [ ]:
def expect_unique(rule, df, keys, notebook):
    duplicates = df.groupBy(*keys).count().filter("count > 1").count()
    print(f"expect {rule}: {duplicates} duplicated keys")
    if duplicates:
        return error_record(notebook, Exception(f"{rule}: {duplicates} duplicated {keys}"))

In [ ]:
def expect_no_orphans(rule, child, child_key, parent, parent_key, notebook):
    """Rows in child whose key has no match in parent: silently dropped by an inner join."""
    orphans = child.join(parent, child[child_key] == parent[parent_key], "left_anti").count()
    print(f"expect {rule}: {orphans} orphan rows")
    if orphans:
        return error_record(notebook, Exception(f"{rule}: {orphans} rows not in {parent_key}"))

In [ ]:
def report(errors, notebook):
    """Log to error_logs, then fail the task. The ml jobs have no check_errors gate."""
    log_errors(errors)
    print(f"{notebook}: {len(errors)} rule(s) failed")
    if errors:
        raise Exception(f"{len(errors)} data quality rule(s) failed, see {INFRA_TABLE}.error_logs")